In [1]:
nodes = [
    'Aw_in1',
    'Aw_in2',
    'B_x_xi',
    'Aw_out',
    'B_ix',
    'B_viii',
    'B_vii',
    'B_vi',
    'B_v',
    'B_iv'
]

node_to_idx = {
    node: i
    for i, node in enumerate(nodes)
}

In [2]:
node_to_idx

{'Aw_in1': 0,
 'Aw_in2': 1,
 'B_x_xi': 2,
 'Aw_out': 3,
 'B_ix': 4,
 'B_viii': 5,
 'B_vii': 6,
 'B_vi': 7,
 'B_v': 8,
 'B_iv': 9}

In [ ]:
import random


def evaluate_infinite_worst_case_expected(
    solution,
    node_to_idx,
    coverage,
    attack_duration,
    target_values,
    adj_matrix,
    attacker_history_length=1,
    rollouts_num=1000,
    rollout_length=1050,
    protect_current_vertex=False,
):

    num_nodes = len(target_values)

    # =========================
    # HELPERS
    # =========================

    def sample_from_distribution(dist):
        items = []
        probs = []

        for x, p in dist.items():
            if p > 1e-12:
                items.append(x)
                probs.append(p)

        return random.choices(items, weights=probs)[0]

    def current_vertex(state):
        return state[0][-1][0]

    def expected_capture_probability(path, target, tau):
        survive_prob = 1.0

        if protect_current_vertex:
            survive_prob *= (1.0 - coverage[path[0]][target])

        for i in range(len(path) - 1):

            pos = path[i]
            next_pos = path[i + 1]

            tau -= adj_matrix[pos][next_pos]

            if tau >= 0:
                survive_prob *= (1.0 - coverage[next_pos][target])
            else:
                break

        return 1.0 - survive_prob

    # =========================
    # HOW MANY ATTACK STARTS
    # =========================

    max_tau = max(attack_duration)
    usable_steps = rollout_length - max_tau - 10 # just to be safe

    observation_rewards = {}

    # =========================
    # MAIN LOOP
    # =========================

    for _ in range(rollouts_num):

        # ----- generate one long rollout -----

        state = sample_from_distribution(solution.stationary)
        state_rollout = [state]

        for _ in range(rollout_length - 1):
            transition = solution.transition[state]
            state = sample_from_distribution(transition)
            state_rollout.append(state)

        vertex_rollout = [
            node_to_idx[current_vertex(s)]
            for s in state_rollout
        ]

        # ----- evaluate every possible attack start -----

        for start in range(1, usable_steps + 1):

            if attacker_history_length == -1:
                observation = tuple(vertex_rollout[:start])
            else:
                observation = tuple(
                    vertex_rollout[
                        max(0, start - attacker_history_length):start
                    ]
                )

            attack_path = vertex_rollout[start - 1:]

            for target in range(num_nodes):

                if target_values[target] == 0:
                    continue

                capture_prob = expected_capture_probability(
                    attack_path,
                    target,
                    attack_duration[target],
                )

                reward = capture_prob * target_values[target]

                key = (observation, target)

                if key not in observation_rewards:
                    observation_rewards[key] = []

                observation_rewards[key].append(reward)

    # =========================
    # WORST CASE
    # =========================

    obs_target_rewards = {}
    obs_target_counts = {}

    for key, vals in observation_rewards.items():

        mean_reward = sum(vals) / len(vals)

        obs_target_rewards[key] = mean_reward
        obs_target_counts[key] = len(vals)

    worst_pair = None
    worst_value = float("inf")
    count = 0

    for pair, value in obs_target_rewards.items():

        if value < worst_value:
            worst_value = value
            worst_pair = pair
            count = obs_target_counts[pair]

    print("\nWorst case:", worst_pair)
    print("Worst-case value:", worst_value)
    print(f"\nWorst-case value in %: {worst_value * 100:.2f}")
    print("\nCount:", count)
    print(obs_target_rewards)
    print(obs_target_counts)
    return worst_value

# Gdynia version 1

protect_current_vertex = false

In [15]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_3.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


evaluate_infinite_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, rollouts_num=1000, rollout_length=1050, protect_current_vertex=False)



Worst case: ((7,), 5)
Worst-case value: 0.0

Worst-case value in %: 0.00

Count: 148144
{((9,), 2): 0.24995190360469827, ((9,), 4): 0.8437215218712029, ((9,), 5): 0.25004809639530173, ((9,), 6): 0.25004809639530173, ((9,), 7): 0.7500480963953018, ((9,), 8): 1.0, ((9,), 9): 0.8437424058323207, ((8,), 2): 0.25000759388574073, ((8,), 4): 0.9375162424778343, ((8,), 5): 0.2499924061142593, ((8,), 6): 0.2499924061142593, ((8,), 7): 0.4999848122285186, ((8,), 8): 0.8749962030571297, ((8,), 9): 0.8958055593993742, ((7,), 2): 0.5, ((7,), 4): 0.9583386772329625, ((7,), 5): 0.0, ((7,), 6): 0.0, ((7,), 7): 0.5, ((7,), 8): 1.0, ((7,), 9): 0.9166613227670375, ((4,), 2): 0.31239653941753276, ((4,), 4): 0.8437523729491392, ((4,), 5): 0.2500493573420945, ((4,), 6): 0.2500493573420945, ((4,), 7): 0.500098714684189, ((4,), 8): 0.8125787819114201, ((4,), 9): 0.9375161360541463, ((3,), 2): 0.12491012931252722, ((3,), 4): 0.9374892408331899, ((3,), 5): 0.24998481058803276, ((3,), 6): 0.24998481058803276, (

0.0

# Gdynia version 1

protect_current_vertex = True

In [16]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_3.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


evaluate_infinite_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, rollouts_num=1000, rollout_length=1050, protect_current_vertex=True)



Worst case: ((9,), 5)
Worst-case value: 0.24979500728862974

Worst-case value in %: 24.98

Count: 197568
{((4,), 2): 0.31261839879245895, ((4,), 4): 1.0, ((4,), 5): 0.2499949348110178, ((4,), 6): 0.2499949348110178, ((4,), 7): 0.4999898696220356, ((4,), 8): 0.9061920669010161, ((4,), 9): 0.9686983983872438, ((3,), 2): 0.5625582066284697, ((3,), 4): 0.9687234274087422, ((3,), 5): 0.2502024578381552, ((3,), 6): 0.2502024578381552, ((3,), 7): 0.625086044581216, ((3,), 8): 0.9374417933715303, ((3,), 9): 0.968725958131719, ((9,), 2): 0.25020499271137026, ((9,), 4): 0.9219126452664399, ((9,), 5): 0.24979500728862974, ((9,), 6): 0.24979500728862974, ((9,), 7): 0.7497950072886297, ((9,), 8): 1.0, ((9,), 9): 1.0, ((8,), 2): 0.24999324976542936, ((8,), 4): 0.9687088657580851, ((8,), 5): 0.25000675023457064, ((8,), 6): 0.25000675023457064, ((8,), 7): 0.7500067502345706, ((8,), 8): 1.0, ((8,), 9): 0.9479708794880622, ((7,), 2): 0.5, ((7,), 4): 0.9582782082782083, ((7,), 5): 0.5, ((7,), 6): 0.5, (

0.24979500728862974

# Gdynia version 2

protect_current_vertex = false

In [17]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_4.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


evaluate_infinite_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, rollouts_num=1000, rollout_length=1050, protect_current_vertex=False)



Worst case: ((7,), 5)
Worst-case value: 0.43175595498623714

Worst-case value in %: 43.18

Count: 205626
{((5,), 2): 0.4609614319985237, ((5,), 4): 0.8758073445285108, ((5,), 5): 0.5195192840007381, ((5,), 6): 0.5390385680014763, ((5,), 7): 1.0, ((5,), 8): 1.0, ((5,), 9): 0.9894330134711201, ((7,), 2): 0.4607248110647486, ((7,), 4): 0.8493399424197329, ((7,), 5): 0.43175595498623714, ((7,), 6): 0.43212069485376364, ((7,), 7): 0.7439161390096584, ((7,), 8): 1.0, ((7,), 9): 0.9467884046764514, ((8,), 2): 0.46071101363828404, ((8,), 4): 0.6925778635412068, ((8,), 5): 0.4607234746531091, ((8,), 6): 0.4608973354790014, ((8,), 7): 0.5492767864348206, ((8,), 8): 0.7900692832424273, ((8,), 9): 0.7386204232472693, ((9,), 2): 0.46010050628566523, ((9,), 4): 0.6974902926159523, ((9,), 5): 0.47577042304886946, ((9,), 6): 0.4759728836830137, ((9,), 7): 0.5237696829278757, ((9,), 8): 0.7749371835714592, ((9,), 9): 0.7507314058008324, ((3,), 2): 0.44580028940600186, ((3,), 4): 0.8326935466552131, ((

0.43175595498623714

# Gdynia version 2

protect_current_vertex = True

In [18]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_4.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


evaluate_infinite_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, rollouts_num=1000, rollout_length=1050, protect_current_vertex=True)



Worst case: ((5,), 2)
Worst-case value: 0.46050531182914073

Worst-case value in %: 46.05

Count: 54501
{((9,), 2): 0.46095242695519634, ((9,), 4): 0.8491485262231382, ((9,), 5): 0.47445511581975486, ((9,), 6): 0.4743343584294025, ((9,), 7): 0.5220154140100041, ((9,), 8): 0.8873610283699362, ((9,), 9): 1.0, ((8,), 2): 0.46057317844422996, ((8,), 4): 0.8459565099547791, ((8,), 5): 0.46112060808079847, ((8,), 6): 0.461024317525832, ((8,), 7): 0.7746182852197564, ((8,), 8): 1.0, ((8,), 9): 0.8688932767794733, ((7,), 2): 0.4605616795855568, ((7,), 4): 0.8492226755735598, ((7,), 5): 0.7161901608693959, ((7,), 6): 0.7160818272114673, ((7,), 7): 1.0, ((7,), 8): 1.0, ((7,), 9): 0.9460139300042847, ((5,), 2): 0.46050531182914073, ((5,), 4): 0.875903653143979, ((5,), 5): 1.0, ((5,), 6): 0.7697473440854297, ((5,), 7): 1.0, ((5,), 8): 1.0, ((5,), 9): 0.9892226748133062, ((3,), 2): 0.7230196880593198, ((3,), 4): 0.9167249424699565, ((3,), 5): 0.46122091536691384, ((3,), 6): 0.46092942981334695, ((

0.46050531182914073

# San Francisco

In [ ]:
import random


def evaluate_infinite_worst_case_expected_sf(
    solution,
    coverage,
    attack_duration,
    target_values,
    adj_matrix,
    attacker_history_length=1,
    rollouts_num=1000,
    rollout_length=1050,
    protect_current_vertex=False,
):

    num_nodes = len(target_values)

    # =========================
    # HELPERS
    # =========================

    def sample_from_distribution(dist):
        items = []
        probs = []

        for x, p in dist.items():
            if p > 1e-12:
                items.append(x)
                probs.append(p)

        return random.choices(items, weights=probs)[0]

    def current_vertex(state):
        return state[0][-1][0]

    def expected_capture_probability(path, target, tau):
        survive_prob = 1.0

        if protect_current_vertex:
            survive_prob *= (1.0 - coverage[path[0]][target])

        for i in range(len(path) - 1):

            pos = path[i]
            next_pos = path[i + 1]

            tau -= adj_matrix[pos][next_pos]

            if tau >= 0:
                survive_prob *= (1.0 - coverage[next_pos][target])
            else:
                break

        return 1.0 - survive_prob

    # =========================
    # HOW MANY ATTACK STARTS
    # =========================

    max_tau = max(attack_duration)
    usable_steps = rollout_length - max_tau - 10 # just to be safe

    observation_rewards = {}

    # =========================
    # MAIN LOOP
    # =========================

    for _ in range(rollouts_num):
        if _ % 100 == 0:
            print(f"Rollout {_}/{rollouts_num}")
        # ----- generate one long rollout -----


        state = sample_from_distribution(solution.stationary)

        vertex_rollout = []

        while len(vertex_rollout) < rollout_length:

            v = current_vertex(state)

            # Ignore intermediate edge states
            if v >= 0:
                vertex_rollout.append(v)

            transition = solution.transition[state]
            state = sample_from_distribution(transition)

        # ----- evaluate every possible attack start -----

        for start in range(1, usable_steps + 1):

            if attacker_history_length == -1:
                observation = tuple(vertex_rollout[:start])
            else:
                observation = tuple(
                    vertex_rollout[
                        max(0, start - attacker_history_length):start
                    ]
                )

            attack_path = vertex_rollout[start - 1:]

            for target in range(num_nodes):

                if target_values[target] == 0:
                    continue

                capture_prob = expected_capture_probability(
                    attack_path,
                    target,
                    attack_duration[target],
                )

                reward = capture_prob * target_values[target]

                key = (observation, target)

                if key not in observation_rewards:
                    observation_rewards[key] = []

                observation_rewards[key].append(reward)

    # =========================
    # WORST CASE
    # =========================

    obs_target_rewards = {}
    obs_target_counts = {}

    for key, vals in observation_rewards.items():

        mean_reward = sum(vals) / len(vals)

        obs_target_rewards[key] = mean_reward
        obs_target_counts[key] = len(vals)

    worst_pair = None
    worst_value = float("inf")
    count = 0

    for pair, value in obs_target_rewards.items():

        if value < worst_value:
            worst_value = value
            worst_pair = pair
            count = obs_target_counts[pair]

    print("\nWorst case:", worst_pair)
    print("Worst-case value:", worst_value)
    print(f"\nWorst-case value in %: {worst_value * 100:.2f}")
    print("\nCount:", count)
    print(obs_target_rewards)
    print(obs_target_counts)
    return worst_value

protect_current_vertex=False

In [26]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_infinite_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, rollouts_num=1000, rollout_length=1050, protect_current_vertex=False)


Rollout 0/1000
Rollout 100/1000
Rollout 200/1000
Rollout 300/1000
Rollout 400/1000
Rollout 500/1000
Rollout 600/1000
Rollout 700/1000
Rollout 800/1000
Rollout 900/1000

Worst case: ((10,), 10)
Worst-case value: 0.0

Worst-case value in %: 0.00

Count: 81625
{((10,), 0): 0.18782235834609495, ((10,), 1): 0.19547932618683, ((10,), 2): 0.19505053598774885, ((10,), 3): 0.19437672281776416, ((10,), 4): 0.19353139356814703, ((10,), 5): 0.19407044410413476, ((10,), 6): 0.18741807044410413, ((10,), 7): 0.19577335375191424, ((10,), 8): 0.1956018376722818, ((10,), 9): 0.19617764165390505, ((10,), 10): 0.0, ((10,), 11): 0.19244104134762635, ((11,), 0): 0.19250457038391225, ((11,), 1): 0.19295679784470315, ((11,), 2): 0.1940825555662465, ((11,), 3): 0.1936399499663235, ((11,), 4): 0.19212931781006445, ((11,), 5): 0.19312036947945732, ((11,), 6): 0.1897046088713557, ((11,), 7): 0.19732512267872607, ((11,), 8): 0.19680554219185992, ((11,), 9): 0.19308188203598575, ((11,), 10): 0.3704512652747041, ((1

0.0

protect_current_vertex=True

In [28]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_infinite_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, rollouts_num=1000, rollout_length=1050, protect_current_vertex=True)


Rollout 0/1000
Rollout 100/1000
Rollout 200/1000
Rollout 300/1000
Rollout 400/1000
Rollout 500/1000
Rollout 600/1000
Rollout 700/1000
Rollout 800/1000
Rollout 900/1000

Worst case: ((7,), 6)
Worst-case value: 0.19044634062096227

Worst-case value in %: 19.04

Count: 78169
{((5,), 0): 0.19288596579404627, ((5,), 1): 0.19499691256085128, ((5,), 2): 0.19255568160604278, ((5,), 3): 0.1917802317733389, ((5,), 4): 0.1940491405431021, ((5,), 5): 1.0, ((5,), 6): 0.194695348737022, ((5,), 7): 0.1927567241552623, ((5,), 8): 0.19399169981475364, ((5,), 9): 0.19483895055789308, ((5,), 10): 0.28891250341054325, ((5,), 11): 0.27404971495038555, ((8,), 0): 0.19413566069514623, ((8,), 1): 0.194334711376512, ((8,), 2): 0.19422753024039197, ((8,), 3): 0.1940284795590262, ((8,), 4): 0.1924820088807227, ((8,), 5): 0.19387536365028327, ((8,), 6): 0.19413566069514623, ((8,), 7): 0.1938141172867861, ((8,), 8): 1.0, ((8,), 9): 0.1924820088807227, ((8,), 10): 0.19085898024804776, ((8,), 11): 0.1918848568366253

0.19044634062096227